Hücre 1 — Kurulum

In [ ]:
!pip -q install xgboost joblib scikit-learn pandas numpy matplotlib

Hücre 2 — Importlar ve klasörler

In [ ]:
import os
import json
import math
import random
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

BASE_DIR = Path("/content/matchfluence_ml")
SEED_DIR = BASE_DIR / "seed_data"
OUTPUT_DIR = BASE_DIR / "outputs"
MODEL_DIR = BASE_DIR / "models"

SEED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("SEED_DIR:", SEED_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)

Hücre 3 — Seed data yükleme

Seed data’yı zip yapıp Colab’a yükleyin. Zip içinde dosyalar şu isimlerde olursa direkt çalışır:

Beklenen minimum dosyalar:

```text
influencer_profiles.json
business_profiles.json
collab_listings.json
instagram_posts.json
```

Opsiyonel davranış dosyaları: `swipes.json`, `matches.json`, `agreements.json`.


In [ ]:
from google.colab import files

print("Seed data zip dosyanı yükle. Örn: seed_data.zip")
uploaded = files.upload()

for filename in uploaded.keys():
    file_path = Path("/content") / filename

    if filename.endswith(".zip"):
        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(SEED_DIR)
        print(f"Extracted: {filename} -> {SEED_DIR}")
    else:
        target = SEED_DIR / filename
        os.rename(file_path, target)
        print(f"Moved: {filename} -> {target}")

print("\nSeed directory files:")
for p in SEED_DIR.rglob("*"):
    if p.is_file():
        print("-", p.relative_to(SEED_DIR))

Hücre 4 — JSON okuma yardımcıları

In [ ]:
def read_json_or_jsonl(path: Path):
    """
    JSON array, JSON object veya JSONL dosyalarını güvenli okur.
    """
    if not path.exists():
        return []

    if path.suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # Olası wrapper formatları:
        # {"influencer_profiles": [...]}
        # {"data": [...]}
        for key in ["data", "items", "records", "influencer_profiles", "business_profiles", "collab_listings", "instagram_posts"]:
            if key in data and isinstance(data[key], list):
                return data[key]

        # Eğer tek kayıt geldiyse listeye çevir
        return [data]

    return []


def find_file(possible_names):
    for name in possible_names:
        matches = list(SEED_DIR.rglob(name))
        if matches:
            return matches[0]
    return None


files_map = {
    "influencers": find_file(["influencer_profiles.json", "influencers.json", "influencer_profiles.jsonl"]),
    "businesses": find_file(["business_profiles.json", "businesses.json", "business_profiles.jsonl"]),
    "listings": find_file(["collab_listings.json", "collaboration_listings.json", "collab_listings.jsonl"]),
    "posts": find_file(["instagram_posts.json", "posts.json", "instagram_posts.jsonl"]),
    "swipes": find_file(["swipes.json", "swipes.jsonl"]),
    "matches": find_file(["matches.json", "matches.jsonl"]),
    "agreements": find_file(["agreements.json", "agreements.jsonl"]),
}

for k, v in files_map.items():
    print(k, "=>", v)

Hücre 5 — Datayı yükle ve kontrol et

In [ ]:
influencers = read_json_or_jsonl(files_map["influencers"]) if files_map["influencers"] else []
businesses = read_json_or_jsonl(files_map["businesses"]) if files_map["businesses"] else []
listings = read_json_or_jsonl(files_map["listings"]) if files_map["listings"] else []
posts = read_json_or_jsonl(files_map["posts"]) if files_map["posts"] else []

swipes = read_json_or_jsonl(files_map["swipes"]) if files_map["swipes"] else []
matches = read_json_or_jsonl(files_map["matches"]) if files_map["matches"] else []
agreements = read_json_or_jsonl(files_map["agreements"]) if files_map["agreements"] else []

print("Influencers:", len(influencers))
print("Businesses:", len(businesses))
print("Collab listings:", len(listings))
print("Instagram posts:", len(posts))
print("Swipes:", len(swipes))
print("Matches:", len(matches))
print("Agreements:", len(agreements))

assert len(influencers) > 0, "Influencer data bulunamadı."
assert len(businesses) > 0, "Business data bulunamadı."
assert len(listings) > 0, "Collab listing data bulunamadı."

print("\nSample influencer:")
print(json.dumps(influencers[0], ensure_ascii=False, indent=2)[:1000])

Hücre 6 — Yardımcı feature fonksiyonları

In [ ]:
def safe_get(d, path, default=None):
    """
    Nested dict güvenli okuma.
    path örn: "location.city"
    """
    cur = d
    for part in path.split("."):
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return default
    return cur


def to_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def normalize_text(x):
    if x is None:
        return ""
    return str(x).strip().lower()


def jaccard(a, b):
    a = set([normalize_text(x) for x in to_list(a) if normalize_text(x)])
    b = set([normalize_text(x) for x in to_list(b) if normalize_text(x)])
    if not a and not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))


def haversine_km(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]:
        return 999.0

    try:
        lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])
    except:
        return 999.0

    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)

    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


def location_score_from_distance(distance_km):
    """
    Dokümandaki mantık: exp(-distance / 10)
    """
    if distance_km is None:
        return 0.0
    return float(np.exp(-distance_km / 10.0))


def tier_to_numeric(tier):
    mapping = {
        "nano": 1,
        "micro": 2,
        "mid": 3,
        "macro": 4,
        "mega": 5
    }
    return mapping.get(normalize_text(tier), 0)


def business_size_to_numeric(size):
    mapping = {
        "küçük": 1,
        "kucuk": 1,
        "small": 1,
        "orta": 2,
        "medium": 2,
        "zincir": 3,
        "chain": 3,
        "large": 3
    }
    return mapping.get(normalize_text(size), 1)


def parse_date_days_ago(date_str):
    if not date_str:
        return 999
    try:
        dt = pd.to_datetime(date_str, utc=True)
        now = pd.Timestamp.utcnow()
        return max(0, int((now - dt).days))
    except:
        return 999


def budget_tier_match(listing_budget_min, listing_budget_max, rate_min, rate_max):
    """
    1.0 = ilan bütçesi influencer rate aralığıyla örtüşüyor
    0.5 = kısmi yakın
    0.0 = uyumsuz
    """
    if None in [listing_budget_min, listing_budget_max, rate_min, rate_max]:
        return 0.0

    try:
        listing_budget_min = float(listing_budget_min)
        listing_budget_max = float(listing_budget_max)
        rate_min = float(rate_min)
        rate_max = float(rate_max)
    except:
        return 0.0

    overlap = max(0, min(listing_budget_max, rate_max) - max(listing_budget_min, rate_min))
    if overlap > 0:
        return 1.0

    listing_mid = (listing_budget_min + listing_budget_max) / 2
    rate_mid = (rate_min + rate_max) / 2
    distance = abs(listing_mid - rate_mid)

    if distance <= 0.25 * max(rate_mid, 1):
        return 0.5

    return 0.0

Hücre 7 — Index ve post özetleri

In [ ]:
business_by_id = {b.get("id"): b for b in businesses}
listing_by_id = {l.get("listing_id") or l.get("id"): l for l in listings}
influencer_by_id = {i.get("id"): i for i in influencers}

posts_by_inf = defaultdict(list)
for p in posts:
    inf_id = p.get("influencer_id")
    if inf_id:
        posts_by_inf[inf_id].append(p)

def summarize_posts_for_influencer(inf_id):
    inf_posts = posts_by_inf.get(inf_id, [])

    if not inf_posts:
        return {
            "avg_likes": 0,
            "avg_comments": 0,
            "avg_saves": 0,
            "post_count": 0,
            "reel_ratio": 0,
            "post_frequency_weekly": 0,
            "last_post_recency_days": 999,
            "hashtags": [],
            "captions": [],
            "mentioned_brands": [],
            "content_categories": [],
            "content_styles": []
        }

    likes = []
    comments = []
    saves = []
    hashtags = []
    captions = []
    mentioned_brands = []
    categories = []
    styles = []
    reel_count = 0
    dates = []

    for p in inf_posts:
        metrics = p.get("metrics", {})
        likes.append(metrics.get("likes", 0) or 0)
        comments.append(metrics.get("comments", 0) or 0)
        saves.append(metrics.get("saves", 0) or 0)

        hashtags.extend(to_list(p.get("hashtags")))
        captions.append(p.get("caption", ""))
        mentioned_brands.extend(to_list(p.get("mentioned_brands")))
        categories.append(p.get("content_category"))
        styles.append(p.get("content_style"))

        if normalize_text(p.get("type")) == "reel" or normalize_text(p.get("media_type")) == "reel":
            reel_count += 1

        if p.get("posted_at"):
            dates.append(pd.to_datetime(p.get("posted_at"), utc=True, errors="coerce"))

    dates = [d for d in dates if not pd.isnull(d)]

    if dates:
        latest = max(dates)
        last_post_recency_days = max(0, int((pd.Timestamp.utcnow() - latest).days))
        oldest = min(dates)
        span_days = max(1, int((max(dates) - oldest).days))
        post_frequency_weekly = len(inf_posts) / max(1, span_days / 7)
    else:
        last_post_recency_days = 999
        post_frequency_weekly = 0

    return {
        "avg_likes": float(np.mean(likes)),
        "avg_comments": float(np.mean(comments)),
        "avg_saves": float(np.mean(saves)),
        "post_count": len(inf_posts),
        "reel_ratio": reel_count / max(1, len(inf_posts)),
        "post_frequency_weekly": post_frequency_weekly,
        "last_post_recency_days": last_post_recency_days,
        "hashtags": hashtags,
        "captions": captions,
        "mentioned_brands": mentioned_brands,
        "content_categories": categories,
        "content_styles": styles
    }

post_summaries = {inf.get("id"): summarize_posts_for_influencer(inf.get("id")) for inf in influencers}

print("Post summary sample:")
sample_inf_id = influencers[0].get("id")
print(sample_inf_id, post_summaries[sample_inf_id])

Hücre 8 — Pair feature extraction

In [ ]:
def extract_features(influencer, listing):
    listing_id = listing.get("listing_id") or listing.get("id")
    business_id = listing.get("business_id")
    business = business_by_id.get(business_id, {})
    post_summary = post_summaries.get(influencer.get("id"), {})

    inf_lat = safe_get(influencer, "location.lat")
    inf_lng = safe_get(influencer, "location.lng")
    biz_lat = safe_get(business, "location.lat")
    biz_lng = safe_get(business, "location.lng")
    distance_km = haversine_km(inf_lat, inf_lng, biz_lat, biz_lng)

    inf_categories = to_list(influencer.get("content_categories"))
    listing_categories = to_list(listing.get("target_categories")) or to_list(business.get("past_collaboration_categories"))
    inf_styles = to_list(influencer.get("content_styles"))
    preferred_styles = to_list(listing.get("preferred_styles")) or to_list(business.get("brand_style"))

    sector_content_match = jaccard(inf_categories, listing_categories)
    if sector_content_match == 0 and inf_categories and listing_categories:
        sector_content_match = 0.15

    audience_demo = influencer.get("audience_demographics", {}) or {}
    loc_dist = audience_demo.get("location_distribution", {}) or {}
    biz_city = safe_get(business, "location.city", "İstanbul")
    audience_location_match = float(loc_dist.get(biz_city, loc_dist.get("İstanbul", 0.0)) or 0.0)

    age_dist = audience_demo.get("age_distribution", {}) or {}
    target_ages = to_list(safe_get(business, "target_audience.age_buckets", []))
    audience_age_overlap = sum(float(age_dist.get(age, 0.0) or 0.0) for age in target_ages)
    audience_age_overlap = min(audience_age_overlap, 1.0)

    gender_pref = normalize_text(safe_get(business, "target_audience.gender_preference", "all"))
    gender_dist = audience_demo.get("gender_distribution", {}) or {}
    if gender_pref in ["female", "male", "other"]:
        audience_gender_match = float(gender_dist.get(gender_pref, 0.0) or 0.0)
    else:
        audience_gender_match = 1.0

    income_dist = audience_demo.get("income_distribution", {}) or {}
    income_groups = to_list(safe_get(business, "target_audience.income_groups", []))
    audience_income_match = sum(float(income_dist.get(group, 0.0) or 0.0) for group in income_groups)
    audience_income_match = min(audience_income_match, 1.0)

    inf_interests = to_list(audience_demo.get("interest_tags"))
    biz_interests = to_list(safe_get(business, "target_audience.interest_tags", []))
    audience_interest_overlap = jaccard(inf_interests, biz_interests)

    budget = listing.get("budget", {}) or {}
    rate = influencer.get("rate_range", {}) or {}
    listing_budget_min = budget.get("min", 0) or 0
    listing_budget_max = budget.get("max", 0) or 0
    rate_min = rate.get("min", 0) or 0
    rate_max = rate.get("max", 0) or 0

    hashtags = post_summary.get("hashtags", []) or []
    captions = " ".join(post_summary.get("captions", []) or [])
    mentioned_brands = post_summary.get("mentioned_brands", []) or []
    business_name_token = normalize_text(business.get("name", "")).replace(" ", "")
    natural_affinity_score = 0.0
    if sector_content_match > 0:
        natural_affinity_score += 0.35
    if business_name_token and business_name_token in normalize_text(captions).replace(" ", ""):
        natural_affinity_score += 0.35
    if mentioned_brands:
        natural_affinity_score += 0.20
    if hashtags:
        natural_affinity_score += min(0.10, len(set(hashtags)) / 100.0)
    natural_affinity_score = min(natural_affinity_score, 1.0)

    follower_count = influencer.get("follower_count", 0) or 0
    following_count = influencer.get("following_count", 0) or 0
    avg_likes = post_summary.get("avg_likes", 0) or 0
    avg_comments = post_summary.get("avg_comments", 0) or 0
    engagement_rate = influencer.get("engagement_rate")
    if engagement_rate is None and follower_count:
        engagement_rate = (avg_likes + avg_comments) / max(follower_count, 1)
    engagement_rate = float(engagement_rate or 0.0)

    last_post_recency_days = post_summary.get("last_post_recency_days", 999) or 999
    recency_score = float(np.exp(-last_post_recency_days / 30.0)) if last_post_recency_days < 999 else 0.0

    return {
        "follower_count_log": math.log10(max(follower_count, 1)),
        "following_ratio": following_count / max(follower_count, 1),
        "engagement_rate": engagement_rate,
        "comment_like_ratio": avg_comments / max(avg_likes, 1),
        "account_age_days": parse_date_days_ago(influencer.get("account_created_at")),
        "post_frequency_weekly": post_summary.get("post_frequency_weekly", 0) or 0,
        "last_post_recency_days": last_post_recency_days,
        "recency_score": recency_score,
        "influencer_tier_numeric": tier_to_numeric(influencer.get("tier")),
        "profile_completion": influencer.get("profile_completion", 0) or 0,
        "verified": 1 if influencer.get("verified") else 0,
        "past_collaboration_count": influencer.get("past_collaboration_count", 0) or 0,
        "reel_post_ratio": post_summary.get("reel_ratio", 0) or 0,
        "business_age_months": business.get("business_age_months", 0) or 0,
        "business_size_numeric": business_size_to_numeric(business.get("size")),
        "listing_budget_min": listing_budget_min,
        "listing_budget_max": listing_budget_max,
        "listing_budget_mid": (float(listing_budget_min) + float(listing_budget_max)) / 2,
        "business_verified": 1 if business.get("verified") else 0,
        "business_past_collab_count": business.get("past_collaboration_count", 0) or 0,
        "deliverable_count": len(to_list(listing.get("deliverables"))),
        "sector_content_match": sector_content_match,
        "location_distance_km": distance_km,
        "location_score": location_score_from_distance(distance_km),
        "audience_location_match": audience_location_match,
        "audience_age_overlap": audience_age_overlap,
        "audience_gender_match": audience_gender_match,
        "audience_income_match": audience_income_match,
        "audience_interest_overlap": audience_interest_overlap,
        "budget_tier_match": budget_tier_match(listing_budget_min, listing_budget_max, rate_min, rate_max),
        "natural_affinity_score": natural_affinity_score,
        "style_match": jaccard(inf_styles, preferred_styles),
        "language_match": 1 if normalize_text(influencer.get("primary_language", "tr")) == "tr" else 0,
        "past_category_experience": jaccard(inf_categories, business.get("past_collaboration_categories", [])),
        "hashtag_overlap": jaccard(hashtags, listing_categories),
        "profile_embedding_similarity": 0.5 + 0.5 * sector_content_match,
        "tier_preference_match": 1 if normalize_text(influencer.get("tier")) in [normalize_text(t) for t in to_list(listing.get("preferred_tiers"))] else 0,
    }

sample_features = extract_features(influencers[0], listings[0])
print("Feature count:", len(sample_features))
print(json.dumps(sample_features, ensure_ascii=False, indent=2)[:1500])



Hücre 9 — Synthetic label üretimi

In [ ]:
def normalize_engagement_rate(x):
    """
    Engagement genelde 0.005 - 0.08 arası.
    0.08 ve üstünü 1'e yakın kabul ediyoruz.
    """
    try:
        x = float(x)
    except:
        return 0.0
    return min(max(x / 0.08, 0.0), 1.0)


def compute_synthetic_score(features):
    score = (
        0.25 * features["sector_content_match"] +
        0.15 * features["location_score"] +
        0.10 * features["audience_location_match"] +
        0.08 * features["audience_age_overlap"] +
        0.08 * features["audience_interest_overlap"] +
        0.10 * features["budget_tier_match"] +
        0.10 * features["natural_affinity_score"] +
        0.05 * normalize_engagement_rate(features["engagement_rate"]) +
        0.04 * features["tier_preference_match"] +
        0.03 * features["style_match"] +
        0.02 * features["recency_score"]
    )

    # Sentetik datada gerçekçiliği artırmak için hafif gürültü
    score += np.random.normal(0, 0.04)
    score = float(np.clip(score, 0.0, 1.0))

    return score


def score_to_label(score):
    if score > 0.70:
        return 2  # iyi match
    elif score > 0.40:
        return 1  # orta match
    else:
        return 0  # kötü match


def label_to_text(label):
    return {
        0: "kotu_match",
        1: "orta_match",
        2: "iyi_match"
    }.get(int(label), "bilinmiyor")

Hücre 10 — Training pair üretimi

In [ ]:
MAX_PAIRS = 5000  # Hackathon için 1000-5000 arası kullanabilirsiniz.

all_pairs = []

for inf in influencers:
    for listing in listings:
        inf_id = inf.get("id")
        listing_id = listing.get("listing_id") or listing.get("id")

        if not inf_id or not listing_id:
            continue

        features = extract_features(inf, listing)
        synthetic_score = compute_synthetic_score(features)
        label = score_to_label(synthetic_score)

        row = {
            "pair_id": f"pair_{len(all_pairs)+1:05d}",
            "influencer_id": inf_id,
            "listing_id": listing_id,
            "synthetic_score": synthetic_score,
            "label": label,
            "label_text": label_to_text(label),
            **features
        }

        all_pairs.append(row)

# Fazla pair varsa sample al
if len(all_pairs) > MAX_PAIRS:
    all_pairs = random.sample(all_pairs, MAX_PAIRS)

df = pd.DataFrame(all_pairs)

print("Dataset shape:", df.shape)
print("Label distribution:")
print(df["label_text"].value_counts(normalize=True).round(3))
print(df["label_text"].value_counts())

df.head()

Hücre 11 — Dataset kaydetme

In [ ]:
training_jsonl_path = OUTPUT_DIR / "training_pairs_xgboost.jsonl"
training_csv_path = OUTPUT_DIR / "training_pairs_xgboost.csv"

df.to_json(training_jsonl_path, orient="records", lines=True, force_ascii=False)
df.to_csv(training_csv_path, index=False)

print("Saved:")
print(training_jsonl_path)
print(training_csv_path)

Hücre 12 — Train / validation / test split

In [ ]:
ID_COLS = ["pair_id", "influencer_id", "listing_id", "label_text"]
TARGET_COL = "label"

drop_cols = ID_COLS + [TARGET_COL, "synthetic_score"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df[TARGET_COL].astype(int)

# Önce train+temp
X_train, X_temp, y_train, y_temp, df_train, df_temp = train_test_split(
    X, y, df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# Sonra validation+test
X_val, X_test, y_val, y_test, df_val, df_test = train_test_split(
    X_temp, y_temp, df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train:", X_train.shape, Counter(y_train))
print("Val:", X_val.shape, Counter(y_val))
print("Test:", X_test.shape, Counter(y_test))
print("Features:", len(feature_cols))
print(feature_cols)

Hücre 13 — XGBoost model eğitimi

In [ ]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    tree_method="hist"
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Training completed.")

Hücre 14 — Evaluation

In [ ]:
def evaluate_split(name, X_split, y_split):
    pred = model.predict(X_split)
    proba = model.predict_proba(X_split)

    acc = accuracy_score(y_split, pred)
    macro_f1 = f1_score(y_split, pred, average="macro")

    print(f"\n=== {name} ===")
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("\nClassification report:")
    print(classification_report(
        y_split,
        pred,
        target_names=["kotu_match", "orta_match", "iyi_match"]
    ))

    return pred, proba


val_pred, val_proba = evaluate_split("Validation", X_val, y_val)
test_pred, test_proba = evaluate_split("Test", X_test, y_test)

Hücre 15 — Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix - Test")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0, 1, 2], ["kotu", "orta", "iyi"])
plt.yticks([0, 1, 2], ["kotu", "orta", "iyi"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

Hücre 16 — Feature importance

In [ ]:
importances = model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
top_n = 20
plot_df = importance_df.head(top_n).iloc[::-1]

plt.figure(figsize=(8, 7))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

Hücre 17 — Reason generator

In [ ]:
def generate_reasons_from_features(features):
    reasons = []
    risks = []

    if features.get("sector_content_match", 0) >= 0.70:
        reasons.append("İçerik kategorisi işletme sektörüyle güçlü uyumlu.")
    elif features.get("sector_content_match", 0) < 0.30:
        risks.append("İçerik kategorisi işletme sektörüyle zayıf uyumlu.")

    if features.get("location_score", 0) >= 0.70:
        reasons.append("Lokasyon yakınlığı güçlü.")
    elif features.get("location_score", 0) < 0.20:
        risks.append("Lokasyon uzaklığı eşleşme kalitesini düşürebilir.")

    if features.get("audience_interest_overlap", 0) >= 0.50:
        reasons.append("Influencer kitlesinin ilgi alanları kampanya hedefiyle örtüşüyor.")

    if features.get("budget_tier_match", 0) >= 1.0:
        reasons.append("Kampanya bütçesi influencer ücret aralığıyla uyumlu.")
    elif features.get("budget_tier_match", 0) == 0:
        risks.append("Bütçe influencer ücret aralığıyla uyumsuz olabilir.")

    if features.get("natural_affinity_score", 0) >= 0.50:
        reasons.append("Geçmiş içeriklerde bu sektöre doğal ilgi sinyali var.")

    if features.get("engagement_rate", 0) >= 0.04:
        reasons.append("Etkileşim oranı güçlü.")

    if features.get("tier_preference_match", 0) == 1:
        reasons.append("Influencer seviyesi kampanyanın tercih ettiği tier ile uyumlu.")

    if not reasons:
        reasons.append("Bazı temel profil sinyalleri kampanya ile kısmi uyum gösteriyor.")

    return reasons[:4], risks[:2]

In [ ]:
def generate_reasons_from_features(features):
    reasons = []
    risks = []

    if features.get("sector_content_match", 0) >= 0.70:
        reasons.append("İçerik kategorisi işletme sektörüyle güçlü uyumlu.")
    elif features.get("sector_content_match", 0) < 0.30:
        risks.append("İçerik kategorisi işletme sektörüyle zayıf uyumlu.")

    if features.get("location_score", 0) >= 0.70:
        reasons.append("Lokasyon yakınlığı güçlü.")
    elif features.get("location_score", 0) < 0.20:
        risks.append("Lokasyon uzaklığı eşleşme kalitesini düşürebilir.")

    if features.get("audience_interest_overlap", 0) >= 0.50:
        reasons.append("Influencer kitlesinin ilgi alanları kampanya hedefiyle örtüşüyor.")

    if features.get("budget_tier_match", 0) >= 1.0:
        reasons.append("Kampanya bütçesi influencer ücret aralığıyla uyumlu.")
    elif features.get("budget_tier_match", 0) == 0:
        risks.append("Bütçe influencer ücret aralığıyla uyumsuz olabilir.")

    if features.get("natural_affinity_score", 0) >= 0.50:
        reasons.append("Geçmiş içeriklerde bu sektöre doğal ilgi sinyali var.")

    if features.get("engagement_rate", 0) >= 0.04:
        reasons.append("Etkileşim oranı güçlü.")

    if features.get("tier_preference_match", 0) == 1:
        reasons.append("Influencer seviyesi kampanyanın tercih ettiği tier ile uyumlu.")

    if not reasons:
        reasons.append("Bazı temel profil sinyalleri kampanya ile kısmi uyum gösteriyor.")

    return reasons[:4], risks[:2]

In [ ]:
model_path = MODEL_DIR / "xgboost_match_ranker.joblib"
metadata_path = MODEL_DIR / "xgboost_match_ranker_metadata.json"
importance_path = OUTPUT_DIR / "feature_importance.csv"

joblib.dump(model, model_path)

metadata = {
    "model_type": "XGBClassifier",
    "task": "influencer_collab_match_classification",
    "classes": {
        "0": "kotu_match",
        "1": "orta_match",
        "2": "iyi_match"
    },
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
    "test_size": int(len(X_test)),
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

importance_df.to_csv(importance_path, index=False)

print("Saved model:", model_path)
print("Saved metadata:", metadata_path)
print("Saved feature importance:", importance_path)

In [ ]:
zip_path = "/content/matchfluence_xgboost_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(BASE_DIR))

    for path in MODEL_DIR.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(BASE_DIR))

print("Created:", zip_path)
files.download(zip_path)